# Ch1 — Why "chunk and pray" fails

A vanilla vector RAG over the Northwind FY2025 annual report. We ask *"What was total revenue?"* The right answer (355.0) lives in a **table cell**; the dense index, scoring lexical overlap, can prefer a prose sentence that *says* "revenue" but carries **no authoritative number** and **no provenance**. We then preview the triple-mediated `RagPipeline`, which returns the ENM-exact figure with a `file:line:char` span.

**Grounded, not guessed.** The tiny vector RAG below runs on CPU with a bag-of-words encoder, so it needs no GPU and no Qwen. The `RagPipeline` listing is marked for CI execution (it loads the trained store and Qwen).

In [ ]:
import os, sys
KNOWLYTIX_SRC = os.environ.get("KNOWLYTIX_SRC", "/home/user/jupyterlab/GMS-knowlytix")
sys.path.insert(0, KNOWLYTIX_SRC)

## The corpus

We load the annual report as plain markdown. Every authoritative number lives in a **table**; the MD&A / Risk / Outlook sections are prose with no figures. We point at `data/annual_report.md` relative to the repo root.

In [ ]:
import pathlib

CODE = os.path.join(os.path.dirname(os.getcwd()), "code") if os.path.basename(os.getcwd()) == "notebooks" else os.getcwd()
DATA = pathlib.Path(os.environ.get("GMS_RAG_DATA", os.path.join(CODE, "data")))
report = (DATA / "annual_report.md").read_text()
lines = report.split("\n")
print(f"{len(lines)} lines, {len(report)} chars")

## Listing 1 — a tiny vector RAG (chunk + embed + top-k)

This is the entire "chunk and pray" recipe: split the document on blank lines, embed each chunk with a deterministic bag-of-words vector, rank by cosine to the question, take top-k. No external model — the failure mode is structural, not a quirk of any one embedder.

In [ ]:
import math, re
from collections import Counter

def chunk(md: str) -> list[tuple[int, str]]:
    """Blank-line paragraph chunks, paired with their 1-based start line."""
    chunks, buf, start = [], [], None
    for i, ln in enumerate(md.split("\n"), start=1):
        if ln.strip() == "":
            if buf:
                chunks.append((start, "\n".join(buf)))
            buf, start = [], None
        else:
            start = i if start is None else start
            buf.append(ln)
    if buf:
        chunks.append((start, "\n".join(buf)))
    return chunks

def embed(text: str) -> Counter:
    """Deterministic bag-of-words vector (the stand-in dense encoder)."""
    return Counter(re.findall(r"[a-z]+", text.lower()))

def cosine(a: Counter, b: Counter) -> float:
    shared = set(a) & set(b)
    dot = sum(a[t] * b[t] for t in shared)
    na = math.sqrt(sum(v * v for v in a.values()))
    nb = math.sqrt(sum(v * v for v in b.values()))
    return dot / (na * nb) if na and nb else 0.0

class VanillaRAG:
    def __init__(self, md: str):
        self.chunks = chunk(md)
        self.index = [embed(c) for _, c in self.chunks]

    def retrieve(self, q: str, top_k: int = 1):
        qv = embed(q)
        scored = [(cosine(qv, cv), self.chunks[i])
                  for i, cv in enumerate(self.index)]
        scored.sort(key=lambda x: x[0], reverse=True)
        return scored[:top_k]

rag = VanillaRAG(report)
print(f"{len(rag.chunks)} chunks indexed")

## The failure: top-k is not relevance

We ask the numeric question. The word "total" in *"total revenue"* lexically matches the **Balance Sheet** block — "**Total** Assets", "**Total** Liabilities" — at least as well as it matches the income-statement Revenue row. A bag-of-words ranker has no notion that "total revenue" is one concept; it just counts overlapping tokens. So the top hit is the **wrong table**: it contains 540.0 / 210.0 / 330.0 and **not** the authoritative 355.0.

Expected (grounded in `data/corpus_facts.md`): the top chunk is the Balance Sheet table near line 45, whose figures do **not** include the answer; the income-statement Revenue cell (355.0 at line 36) is out-ranked.

In [ ]:
question = "What was total revenue?"
hits = rag.retrieve(question, top_k=3)
for rank, (score, (line_no, text)) in enumerate(hits, start=1):
    snippet = text.replace("\n", " ")[:80]
    print(f"#{rank}  score={score:.3f}  line={line_no}  {snippet!r}")

top_score, (top_line, top_text) = hits[0]
has_answer = bool(re.search(r"\b355(?:\.0)?\b", top_text))
numbers = re.findall(r"\d+\.\d+", top_text)
print()
print(f"top chunk contains the answer (355): {has_answer}")
print(f"numbers in the top chunk (ambiguous): {numbers}")
print("cell-level provenance:                None  "
      "(chunk spans many rows; no file:line:char for one cell)")

**What this means.** The vanilla RAG would now hand that *wrong* table to an LLM and ask it to "answer from context." The model must pick a number from 540.0 / 210.0 / 330.0 — a **silently wrong** figure — or, when handed the right table, *parse* one cell out of eight with **no cell-level provenance**. There is no guard against either error. These are three of the four failure modes from the chapter thesis: hallucinated retrieval (wrong chunk ranks first), no provenance (chunk-level line range only), and silently wrong numbers (parsed from text). The fourth — unverifiable, "a model judging a model" — appears the moment you add an LLM grader to this same loop (Ch10).

Cross-reference: the agent book's Ch9 (memory tiers) treats this dense index as the *lowest-trust* tier; this book quarantines it entirely (Ch14).

## Listing 2 — the same question through `RagPipeline` (GMS box)

**Marked for CI execution.** This cell loads the trained store and the local Qwen synthesizer; do not run it during authoring (shared GPU). It returns the **ENM-exact** total revenue with a `file:line:char` provenance span. The expected values come from `data/corpus_facts.md`: `lookup_enm("income_statement", "Revenue/FY2025") == 355.0`, retrieved as the triple `('revenue', 'has_fy2025', '355.0')` with a span into the income-statement table.

In [ ]:
# === CI-ONLY: loads the trained store + Qwen. Do not run during authoring. ===
import torch
from knowlytix.knowledge.store import GMSExpertStore
from knowlytix.knowledge.config import DocGMSConfig
from knowlytix.knowledge.llm_backend import LocalTransformersBackend
from knowlytix.knowledge.rag import RagConfig, RagPipeline
from knowlytix.knowledge.geode import QWEN_3B

STORE_PATH = str(DATA / "gms_annual_report_store")
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

store = GMSExpertStore(DocGMSConfig(store_path=STORE_PATH), device=device)
assert store.load(), f"no trained store at {STORE_PATH}"

# ENM gives the authoritative number directly, byte-exact, no parsing.
enm_total = store.lookup_enm("income_statement", "Revenue/FY2025")
print(f"lookup_enm -> {enm_total}")   # 355.0

qwen = LocalTransformersBackend(QWEN_3B, device=str(device))
pipe = RagPipeline.from_store(store, RagConfig(llm=qwen))
ans = pipe.query("What was total revenue?")
print("answer  :", ans.answer)
print("decision:", ans.decision, "| route:", ans.route,
      "| verified:", ans.verified)
for f in ans.sources:
    print(f"  fact: ({f.head}, {f.relation}, {f.tail})  "
          f"src={f.source}  @ {f.location}")

**What this means.** The triple-mediated route never parses a number out of prose: the figure comes from Exact Numerical Memory (`lookup_enm`), and the retrieved triple carries a `file:line:char` span back to the exact table cell. The answer is *grounded* and *verifiable* — and when no triple binds (the MD&A/Risk/Outlook prose), the pipeline **abstains** instead of guessing (Ch11).

**Honest limit.** Triple-mediation only helps where triples exist. The four prose sections of this report carry zero triples (coverage_ratio = 0.56 in `corpus_facts.md`); a question answerable only from prose will *abstain*, not answer, unless you opt into the distrusted dense fallback (Ch14). Chunk-and-pray would have *guessed*; this is the trade the book argues for.

## Exercise

Break the vector RAG on a **second** numeric question. We ask for net income (70.0 per `corpus_facts.md`). Here top-1 *is* the income-statement table — but the chunk holds **eight** numbers across four rows and two years, so there is no way to return a single, provenance-backed figure. Retrieving the right table is not the same as answering the question.

In [ ]:
# Exercise solution: a second numeric question, same structural failure.
q2 = "What was net income?"
hit2 = rag.retrieve(q2, top_k=1)[0]
score2, (line2, text2) = hit2
print(f"top hit: line={line2}  score={score2:.3f}")
print(text2.replace(chr(10), " ")[:90])

# The vanilla RAG offers no ENM and no cell-level span. Even if the income-
# statement table is retrieved, the number must be *parsed* from text, and
# the chunk spans many rows -- there is no provenance for the 70.0 cell.
parsed_numbers = re.findall(r"\d+\.\d+", text2)
print("numbers visible in the chunk (ambiguous, unparsed):", parsed_numbers)
assert len(parsed_numbers) != 1, (
    "Either prose (0 numbers) or a multi-row table (>1 number): in neither\n"
    "case can the vanilla RAG return ONE provenance-backed figure."
)
print("=> no single, provenance-backed number is recoverable. Failure confirmed.")

## Self-check

The chapter's claim: a top-k vector retriever, ranking the numeric question by lexical overlap, does **not** isolate the authoritative table cell — so it cannot return a single provenance-backed figure. We assert that directly (CPU-only; no store, no Qwen).

In [ ]:
# Self-check (CPU-only): the vanilla top-1 for the numeric question is NOT a
# clean, single-figure table cell with provenance -> chunk-and-pray fails.
top1_score, (top1_line, top1_text) = rag.retrieve("What was total revenue?", top_k=1)[0]
clean_table_cell = ("|" in top1_text
                    and len(re.findall(r"\d+\.\d+", top1_text)) == 1)
assert not clean_table_cell, (
    "Vanilla RAG unexpectedly isolated a single-figure cell; "
    "the chunk-and-pray failure did not reproduce."
)
print("OK: vanilla vector RAG returns no single provenance-backed figure.")
print("    The triple-mediated route (Listing 2) returns 355.0 with a span.")